In [1]:
import os

# TABPFN_TOKEN comes from the gitignored .env, which VS Code loads into the kernel.
assert "TABPFN_TOKEN" in os.environ, "Add TABPFN_TOKEN to .env and restart the kernel."

In [2]:
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from tabpfn import TabPFNRegressor

X, y = fetch_california_housing(return_X_y=True)
# Subset of the data for speed reasons
X = X[:1000]
y = y[:1000]
print(X.shape)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

clf = TabPFNRegressor()
clf.fit(X_train, y_train)
prediction_probabilities = clf.predict(X_test)
predictions = clf.predict(X_test)
print("R2 Score", r2_score(y_test, predictions))

(1000, 8)
R2 Score 0.8509269927986258


In [ ]:
import jax.numpy as jnp

from shapiq.masker import BaselineMasker
from shapiq.players import FinitePlayers
from shapiq.xai.masked import MaskedGame

explanation_game = MaskedGame(
    model=clf,
    players=FinitePlayers(ids=jnp.arange(X.shape[1])),
    x_explain=X_test[:2][None, ...],  # (*batch, n_inputs, n_features)
    background=X_train[:10][:, None, :],  # (n_background, *batch,  n_features)
    input_shape=X_train.shape[1:],
    masker=BaselineMasker(),
    batch_size=128,
)

In [23]:
from shapiq.game.exact import ExactComputer
from shapiq.index import ShapleyValueIndex

shapley_index = ShapleyValueIndex()
computer = ExactComputer.from_game(explanation_game)

In [ ]:
results = computer(shapley_index)
print(results)
print("WE have computed in total", results.shape[0], "explanations.")

WE have computed in total 10 explanations.


In [25]:
from shapiq.masker.base import MarginalMasker

explanation_game = MaskedGame(
    model=clf,
    players=FinitePlayers(ids=jnp.arange(X.shape[1])),
    x_explain=X_test[:5],
    background=X_train[:5],
    input_shape=X_train.shape[1:],
    masker=MarginalMasker(),
    batch_size=128,
)

In [26]:
from shapiq.game.exact import ExactComputer
from shapiq.index import ShapleyValueIndex

shapley_index = ShapleyValueIndex()
computer = ExactComputer.from_game(explanation_game)

In [ ]:
results = computer(shapley_index)
print(results)
print("WE have computed in total", results.shape[0], "explanations.")

WE have computed in total 5 explanations.


## Recontexualize

In [ ]:
from shapiq.xai.recontextualized import RecontextualizedGame

explanation_game = RecontextualizedGame(
    model=clf,
    players=FinitePlayers(ids=jnp.arange(X.shape[1])),
    x_explain=X_test[0],
    background=X_train,
    input_shape=X_train.shape[1:],
    y_background=y_train,
)

In [30]:
from shapiq.game.exact import ExactComputer
from shapiq.index import ShapleyValueIndex

shapley_index = ShapleyValueIndex()
computer = ExactComputer.from_game(explanation_game)

In [33]:
results = computer(shapley_index)
print(results)
print("WE have computed in total", results.shape, "explanations.")

AdditiveMoebiusGame(values=Array([ 0.00517112,  0.1325748 ,  0.6587886 ,  0.06487745,  0.00794843,
        0.12569976, -0.22611189,  0.26644468], dtype=float32), players=FinitePlayers(ids=Array([0, 1, 2, 3, 4, 5, 6, 7], dtype=int32), players=None, latent_n_players=8, n_players=8), interactions=Array([[ True, False, False, False, False, False, False, False],
       [False,  True, False, False, False, False, False, False],
       [False, False,  True, False, False, False, False, False],
       [False, False, False,  True, False, False, False, False],
       [False, False, False, False,  True, False, False, False],
       [False, False, False, False, False,  True, False, False],
       [False, False, False, False, False, False,  True, False],
       [False, False, False, False, False, False, False,  True]],      dtype=bool), additivity=1)
WE have computed in total () explanations.
